教学文件和生成内容保存在 /root/gpufree-data/PATHAK/tutorials/。选择 Python (pathak) 内核，按顺序执行。

PROJECT_ROOT 指向项目根目录，TUTORIALS_ROOT 指向教学目录。BioBERT 完整模型位于 models/biobert_pubmed_sentence/，Java JAR 和分句资源位于 resources/。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/root/gpufree-data/PATHAK")
TUTORIALS_ROOT = PROJECT_ROOT / "tutorials"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import importlib
import pathak_runtime
importlib.reload(pathak_runtime)
from pathak_runtime import configure
configure()
print("Project:", PROJECT_ROOT)
print("Tutorials:", TUTORIALS_ROOT)
print("Python:", sys.executable)


Project: /root/gpufree-data/PATHAK
Tutorials: /root/gpufree-data/PATHAK/tutorials
Python: /opt/conda/envs/pathak/bin/python


# Pathak: Example

<div class="alert alert-block alert-info">
    <h2>🐍Load Python libraries</h2>
</div>

In [2]:
import os
from pathlib import Path
import pickle
import os.path

import requests
import pickle
import re
import json
import typing


import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

import nlu
import nltk
# nltk.download('punkt')
from nltk.tokenize import word_tokenize  
from nltk import FreqDist, sent_tokenize, word_tokenize # $ pip install nltk

from bs4 import BeautifulSoup

from IPython.display import display
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
# pio.renderers
pio.renderers.default = "svg"

from itertools import chain
from tqdm.notebook import tqdm, trange
import warnings
import session_info
import time
start = time.time()

<div class="alert alert-block alert-warning">
    <h2><b>Environment setup: </b>Make sure java is loaded</h2>
</div>

#### Load Java 8 and Set Java home directory

In [3]:
# Load Java 8 required by nlu and Pyspark
!java -version

openjdk version "1.8.0_472"
OpenJDK Runtime Environment (Zulu 8.90.0.19-CA-linux64) (build 1.8.0_472-b08)
OpenJDK 64-Bit Server VM (Zulu 8.90.0.19-CA-linux64) (build 25.472-b08, mixed mode)


In [4]:
# Set Java home directory cheaha
print("Java home:", os.environ["JAVA_HOME"])

Java home: /opt/conda/envs/pathak


```
print(os.environ["JAVA_HOME"])
```

<div class="alert alert-block alert-info">
    <h2>Set spark environment varibales</h2>
</div>

In [5]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

加载第二课结果：tutorials/GO_BP_Ontology_corups_Filtered.tsv 提供 GO、Name、Definition、Depth；tutorials/predictions_full_BP.pickle 提供 document 和 sentence_embedding_biobert。两者的定义和向量必须逐行对应。


In [6]:
%%time

df = pd.read_csv(TUTORIALS_ROOT / "GO_BP_Ontology_corups_Filtered.tsv", sep="\t")

with open(TUTORIALS_ROOT / "predictions_full_BP.pickle", 'rb') as handle:
    predictions = pickle.load(handle)
display(predictions)
# GO 表提供术语信息，pickle 提供向量；二者必须逐行对应。
normalize_text = lambda text: " ".join(str(text).split())
assert len(df) == len(predictions), "GO 表与向量行数不一致"
assert df.Definition.map(normalize_text).tolist() == predictions.document.map(normalize_text).tolist(), "GO 定义与向量顺序不一致"


,document,sentence_embedding_biobert
0,The repair of single strand breaks in DNA.,"[-0.05223826691508293, -0.028281446546316147, ..."
1,Repair of such breaks is mediated by the same ...,"[0.02170223370194435, -0.04093412682414055, -0..."
2,The chemical reactions and pathways involving ...,"[0.1769087016582489, 0.0583542063832283, -0.13..."
3,The chemical reactions and pathways resulting ...,"[0.10049676150083542, -0.038804393261671066, -..."
4,The chemical reactions and pathways resulting ...,"[0.09165223687887192, -0.011721471324563026, -..."
...,...,...
5112,Any process that activates or increases the fr...,"[0.08211275935173035, -0.05176454782485962, -0..."
5113,Any process that activates or increases the fr...,"[0.015734165906906128, 0.04585925489664078, -0..."
5114,The chemical reactions and pathways involving ...,"[0.10708004236221313, 0.012329170480370522, -0..."
5115,The chemical reactions and pathways resulting ...,"[0.1135878786444664, -0.1308894157409668, -0.3..."


CPU times: user 41.5 ms, sys: 20.2 ms, total: 61.7 ms
Wall time: 61 ms


In [ ]:
%%time
## Calculate dinstance between all pairs of sentences in DF 
from pathak_runtime import start_spark, embedding_column, configure_biobert_pipeline, check_embedding_manifest
spark = start_spark()  # 启动包含本地 Spark NLP JAR 的运行环境。
e_col = embedding_column(predictions)  # 找到保存向量的列名。
stored_model_path = str(TUTORIALS_ROOT / "pipe_full_BP")
embedding_manifest = check_embedding_manifest(
    TUTORIALS_ROOT / "GO_BP_Ontology_corups_Filtered.tsv",
    TUTORIALS_ROOT / "predictions_full_BP.pickle",
    TUTORIALS_ROOT / "pipe_full_BP",
)
pipe = configure_biobert_pipeline(nlu.load(path=stored_model_path))

get_sim_df_for_string()：用同一 BioBERT 编码查询句，再与已保存的 GO 向量计算余弦相似度。sim_score 表示文本相似度。当前实现形成 N×N 矩阵，实际只需要 1×N，后续可以优化。

编码协议现为 masked_mean_parallel_sentence_v1，默认 batchSize=16；全局 intra-op=8、inter-op=8，句内上限 2、OMP=2。图内并行计算各句，共享权重，保留旧管线的运行时算子顺序；按 input_mask 排除 PAD，保留 CLS/SEP。加载前核对语料、向量、模型和保存管线的 manifest，避免使用不配套的产物。编码检查、全库验收与实际速度见 resources/checks/reports/BIOBERT_PERFORMANCE_REPORT.md 和 resources/checks/embedding-performance-global8-validation.json。已有内核需重启，并重新加载向量和管线；本轮未验证完整论文处理流程，检索阈值与关系证据仍需单独评估。


In [ ]:
def get_sim_df_for_string(predictions, e_col, string_to_embed, pipe=pipe):

    """
    Creates a Dataframe which has a `sim_score` column which describes the similarity with the `string_to_embed` variable.

    Args:
      predictions: A DataFrame of predictions.
      e_col: The name of the column in `predictions` that contains the embeddings.
      string_to_embed: The string to embed.
      pipe: An NLU / Spark NLP pipeline object.

    Returns:
      A DataFrame with an additional `sim_score` column.
    """

    # Put predictions vectors in matrix.
    embed_mat = np.array([x for x in predictions[e_col]])  # 把 GO 向量整理成矩阵

    # Embed string input string.  # 编码查询句子
    query_predictions = pipe.predict(string_to_embed, output_level="document")
    embedding = query_predictions.iloc[0][embedding_column(query_predictions)]

    # Replicate embedding for input string.
    m = np.array([embedding,] * len(df))

    # Compute cosine similarity.
    sim_mat = cosine_similarity(m, embed_mat)

    # Write sim score.
    df['sim_score'] = sim_mat[0]  # 用余弦相似度比较查询向量与 GO 向量，并把结果放进：df["sim_score"]

    # Return df.
    return df

get_sim_df_for_string(predictions, e_col, "Test test" )

query_go()：按相似度排序，取 Top 条定义记录，再保留 sim_score > sim_cutoff 的结果。同一 GO 可以对应多条定义记录。


In [ ]:
%%time

def query_go(query: str, Top: int = 3, sim_cutoff: float = 0.92, Send: str = "ALL") -> typing.Union[list, bool]:

    """
    Queries the Gene Ontology (GO) database for terms that are similar to the given query.

    Args:
      query: The query string.
      Top: The number of top results to return.
      sim_cutoff: The similarity cutoff.
      Send: The type of results to send.

    Returns:
      A list of GO terms that are similar to the query, or False if no results were found.
    """

    # Get the similarity matrix between the query and the GO terms.
    sim_df = get_sim_df_for_string(predictions, e_col, query)

    # Sort the similarity matrix by similarity score.
    df_res = sim_df.sort_values('sim_score', ascending=False).iloc[:Top]

    # Filter the results by similarity score.
    df_res = df_res.loc[df_res['sim_score'] > sim_cutoff]

    # If no results were found, return False.
    if not len(df_res):
        return False

    # Return the results.
    if Send == "ALL":
        return [tuple(i) for i in (df_res[["GO", "Name", "Definition", "Depth", "sim_score"]].values.tolist())]
    elif Send == "GO":
        return [tuple(i) for i in (df_res[["GO", "sim_score"]].values.tolist())]
    elif Send == "Name":
        return [tuple(i) for i in (df_res[["Name", "sim_score"]].values.tolist())]
    elif Send == "Def":
        return [tuple(i) for i in (df_res[["Definition", "sim_score"]].values.tolist())]
    elif Send == "Depth":
        return [tuple(i) for i in (df_res[["Depth", "sim_score"]].values.tolist())]
    else:
        raise ValueError("Invalid value for `Send`.")
        
# query_go('Gonadoblastoma is defined as an abnormal growth of germ cells and sex cord elements on the gonad.')
# query_go(' regulate signal transduction events generated by G-protein coupled receptors', sim_cutoff=0.80)

<div class="alert alert-block alert-info">
    <h3><b>Function:</b> Lower case and uppercase lambda function</h3>
    The given Python code defines two lambda functions: Lowerset and Upperset. Both functions take a set as input, convert all elements to lowercase or uppercase respectively, and return a new set containing the transformed elements.
</div>


In [ ]:
# Define a lambda function LowerSet that transforms all elements in a set to lowercase
Lowerset = lambda Set: set([i.lower() for i in Set])

# Define a lambda function UpperSet that transforms all elements in a set to uppercase
Upperset = lambda Set: set([i.upper() for i in Set])


加载拟南芥基因词典 Auxiliary_Files/1.2_ATH_Gene_all.lst，并移除容易与普通单词混淆的名称。当前示例使用拟南芥资源。


In [ ]:
# Read gene names from file and create a list of unique gene names
Gene_list = list(set([i for i in open(TUTORIALS_ROOT / "Auxiliary_Files/1.2_ATH_Gene_all.lst").read().splitlines() if len(i)]))

# Print the number of genes and the first five genes in the list
print(f"Total gene names in Arabidopsis : {len(Gene_list)} \n {Gene_list[:5]}")

# Remove certain terms from the list of genes
Gene_list = set(Gene_list) - set(["AND", "AN", "CAN", "NOT", "HAS", "FACT", 
                                  "MIN", "LATE", "MAIN", "RING", "BP", 
                                  "POLAR", "CO", "NIH", "LAB", "GI", "NSF", 
                                  "WAS", "MICE"])

# Convert the set of genes to a list, and then convert it to uppercase using the `Upperset` function
Gene_list = Upperset(set(list(Gene_list)))

# Print the number of gene IDs and the last five gene IDs in the list
print(f"Total IDs in ATH : {len(Gene_list)} \n {list(Gene_list)[-5:]}")


para_pass()：段落分句、分词，统一大小写后与基因词典取交集；找到时返回基因名称集合，找不到时返回 False。


In [ ]:
%%time

def para_pass(text: str, Print_list: bool = False):
    """
    Takes a text paragraph and returns a boolean value indicating whether or not any genes were found in the text.

    Args:
    text: The text paragraph to analyze.
    Print_list: If True, the list of genes found in the text will be printed.

    Returns:
    A boolean value indicating whether or not any genes were found in the text.
    """

    # Tokenize the text into words.
    words = chain.from_iterable(map(word_tokenize, sent_tokenize(text)))

    # Create a frequency distribution of the words.
    freq = FreqDist(map(str.casefold, words))

    # Convert the frequency distribution to a set of words.
    word_bag = Upperset(freq)

    # Intersect the set of words with the set of genes.
    intersection = word_bag.intersection(Gene_list)

    # If the intersection is not empty, return True.
    if intersection:
        if Print_list:
            print(intersection)
        return intersection

    # Otherwise, return False.
    return False

Plot_data()：展示论文信息表、gene–GO 网络和结果表。网络边来自段落中的基因与 GO 命中组合，需要回到证据句核查。


In [ ]:
def Plot_data(Data) -> nx.Graph:
    """
    Plots the data in the given `Data` dictionary.

    Args:
    Data: A dictionary containing the following keys:
      * `Meta_info`: A dictionary containing the metadata for the paper.
      * `Search_info`: A list of dictionaries, each of which contains the results of a search for a gene.

    Returns:
    A networkx graph of the data.
    """

    # Get the metadata for the paper.
    Meta = Data[0]["Meta_info"]
    #print(Meta)
    values = [['<b>Paper Title</b>', 'Species', 'pmid', 'PMC', 'DOI'],
              ['<b>' + Meta['Title'] + "</b>", Meta['Species'], Meta['pmid'], Meta['pmc'], Meta['DOI']]]

    fig = go.Figure(data=[go.Table(
      columnorder = [1,2],
      columnwidth = [100,400],
      header = dict(
        values = [['<b>Article information</b>'],
                      ['<b>Description</b>']],
        line_color='darkslategray',
        fill_color='#144B39',
        align=['left','center'],
        font=dict(color='white', size=12),
        height=20
      ),
      cells=dict(
        values=values,
        line_color='darkslategray',
        fill=dict(color=['#FFD400', 'white']),
        align=['left', 'left'],
        font_size=12,
        height=20)
        )
    ])

    fig.update_layout(width=800, height=175)
    fig.update_layout(margin = dict(t=0, l=5, r=0, b=0))
    fig.show()
    ########################

    ########################
    Dic_list = []

    for i in range(1, len(Data)-1):
        #print(i)
        Res = Data[i]["Search_info"]

        if len(Res):
            for s in Res:
                for g in Data[i]["Genes"].split(";"):
                    Dic = {"Genes":g, "GO":s["GO"], "Name":s["GO_Name"], "Score":s["Score"], "Depth":s["Depth"]}
                    Dic_list.append(Dic)

    df = pd.DataFrame(Dic_list)
    df_uniq = df.drop_duplicates(keep="first")
    if df_uniq.empty:
        print('No Arabidopsis genes found or GO matched.\n')
        return


    ######################
    # Plot Bipartite Graph
    ######################
    G = nx.from_pandas_edgelist(df_uniq, "Genes", "GO", ['Name', 'Score'])
    G.name = "PhenoMiner"
    color_map = []
    for node in G:
        if node.startswith("GO:"):
            color_map.append('#80BC00')
        else: 
            color_map.append('#FFD400')


    try:
        top = nx.bipartite.sets(G)[0]
    except:
        print("Warning: AmbiguousSolution")
        return G
    
    pos = nx.bipartite_layout(G, top)

    nx.draw(G, pos=pos, node_color=color_map, with_labels=True, 
            verticalalignment='bottom', node_size=100, font_size = 12)

    ######################


    fig = go.Figure(data=[go.Table(
    header=dict(values=list(df_uniq.columns),
                fill_color='#144B39', 
                font=dict(color='white', size=12),
                line_color='#FFD400',
                align='left'),
    cells=dict(values=[df_uniq.Genes, df_uniq.GO, df_uniq.Name, df_uniq.Score, df_uniq.Depth],
               fill_color='white', 
               font=dict(color='#144B39', size=12), 
               line_color='#80BC00',
               align='left'))
                         ])

    fig.update_layout(margin = dict(t=0, l=5, r=5, b=0))
    fig.show()
    
    return G

read_pubmed() 串联完整流程。

1. Offline 读取本地 XML；PMC 按 PMCID 在线获取内容。
2. 提取标题、PMID、PMCID、DOI 和物种名称匹配结果。Organism 只是名称匹配，不是严格的物种过滤器。
3. 用 para_pass() 筛选含词典基因的段落。
4. 逐句调用 query_go()，保存证据句、GO、定义、Depth 和 Score。
5. 返回列表：首项是 Meta_info，后面是段落的 Genes、Para 和 Search_info。Print=False 关闭整份 JSON 打印；日志写入 tutorials/temp.del。

源码注意：for q in query_res 中使用 query_res[0]，会重复第一条命中；Plot_data() 的 range(1, len(Data)-1) 会漏掉最后一个段落。


In [ ]:
%%time
title_meta = ''


def read_pubmed(File, Method = None, sim_cutoff = 0.92, Organism = ['Arabidopsis', 'thaliana', 'A. thaliana', 'thaliana', 'A. thaliana',], 
                Print = True):
    """
      Reads a PubMed file and returns a list of dictionaries, each of which contains the following keys:
        * `Meta_info`: A dictionary containing the metadata for the paper.
        * `Para_info`: A list of dictionaries, each of which contains the information for a paragraph.

      Args:
        File: The path to the PubMed file.
        Method: The method to use to read the file. Can be `"Offline"` to read an offline XML file, or `"PMC"` to read an online file using the given PMC ID.
            Method="Offline"  # 读取本地 XML
            Method="PMC"      # 按 PMCID 在线获取内容
        sim_cutoff: The similarity cutoff to use when searching for genes in the text.
        Organism: A list of strings containing the names of the organisms to search for.
        Print: A boolean value indicating whether or not to print status messages.

      Returns:
        A list of dictionaries, each of which contains the metadata and information for a paragraph in the PubMed file.
    """


    
    #####################################################
    # Check if Whole body is accessible
    #####################################################
    fh = open(TUTORIALS_ROOT / "temp.del", "w")
    if not Method:
        warnings.warn("""Warning Message: 
                      Method not set
                      Options:
                      \tOffline : Reads offline xml file.
                      \t\t  Example : read_pubmed('./manuscript/PMC004xxxxxx/PMC4000261.xml'
                      \tOnline  : Reads online file using given PMC ID
                      \t\t  read_pubmed('4304705')
                      """)
        return False
    
    
    #####################################################
    # Use Beautifulsoup for split doucment in para
    #####################################################    
    if Method == "Offline":
        with open(File, 'r') as f:
            data = f.read()
        Bs_data = BeautifulSoup(data, "html.parser")
    
    if Method == "PMC":
        if File.startswith("PMC"):
            File = File.replace("PMC", "")
        URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pmc&id="+ str(File) +"&tool=my_tool&email=my_email@example.com"
        data = requests.get(URL)
        Bs_data = BeautifulSoup(data.text, "html.parser")
        if not Bs_data.body and Bs_data.abstract:
            warnings.warn("Full text is not accessible, searching is limited to 'Abstract' only!!!\n Try other method type like URL or Offline XML.")
            
    #####################################################
    # Mate info from the document
    #####################################################        
    # Organism
    Species = []
    for Sp in Organism:
        Sp_text = Bs_data(text=re.compile(Sp))
        if Sp_text:
            #print(Sp)
            Species.append(Sp)
    Species = "; ".join(Species)
    
    if not len(Species):
        Species = "**"
        #return False
    if Print:
        print(f"Species = {Species}")
    
    
    
    # Metadata
    Title = ''
    pmid = ""
    pmc = ""
    doi = ''
    b_meta = Bs_data.find_all("article-id")
    for m in b_meta:
        #print(m)
        if "pmid" in m.attrs.values():
            pmid = m.text
            #print(m.text)

        if "pmc" in m.attrs.values():
            pmc = m.text
            #print(m.text)

        if "doi" in m.attrs.values():
            doi = m.text
            #print(m.text)


    Title = str(Bs_data.find('title-group').contents[1]).replace("<article-title>", "").replace("</article-title>", "")
    #print(Title)


    b_unique = Bs_data.find_all('p')

    Unkown_para_num = 1

    List_dic = [{"Meta_info":{"Title":Title, "Species":Species, "pmid":pmid, "pmc":pmc, "DOI":doi}}]

    #####################################################
    # Process para 
    #####################################################    
    for i in b_unique:
        #print(i.name, i.attrs,i.text, "##########")
        para_ID = ""
        #Text = i.text
        if 'id' in i.attrs:
            #print(i.attrs['id'], "||||||||||||||||")
            para_ID = i.attrs['id']
        else:
            #print("U" + str(Unkown_para_num), "##########")
            para_ID = "U" + str(Unkown_para_num)
            Unkown_para_num += 1
        #####################################################
        # Split para into lines
        #####################################################
        if para_pass(i.text):
            dic = {"Para_number":para_ID, "Genes":";".join(para_pass(i.text)), "Para":i.text}
            #dic = {"Species":Species, "pmid":pmid, "pmc":pmc, "Para_number":para_ID, "Genes":";".join(para_pass(i.text)), "Para":i.text, "DOI":doi}
            
            #print(pmid, pmc, para_ID, ";".join(para_pass(i.text)), i.text, doi, sep="\t")
            #print("|_________________|", json.dumps(dic, indent=4, default=str), i.text, file=fh)
            Search_info = []
            for sent in sent_tokenize(i.text):
                query_res = query_go(sent, sim_cutoff = sim_cutoff)
                
                if query_res:
                    #print(query_res)
                    for q in query_res:
                        query_dic = {"Sentance":sent,
                                     "GO":query_res[0][0], 
                                     "GO_Name":query_res[0][1], 
                                     "Definition":query_res[0][2], 
                                     "Depth":query_res[0][3],
                                     "Score":query_res[0][4]
                                    }
                        Search_info.append(query_dic)
                    print(sent, dic["Genes"], file=fh)
                    print(query_dic, file=fh)
                    print(">" * 10, file=fh)
            print("#" * 10, file = fh)
            dic["Search_info"] = Search_info
            List_dic.append(dic)
            
        
    if len(List_dic):
        if Print:
            print(json.dumps(List_dic, indent=4, default=str))
            #print(json.dumps(List_dic, indent=4, default=str), file=fh)
        return List_dic
    
    fh.close()
    return False



#### Call function to predict GOs
<div class="alert alert-block alert-danger">
    <h2><b>Example 1 (Bad):</b> PMC1459476</h2>
    Two Isoforms of a Divalent Metal Transporter (DMT1) in Schistosoma mansoni Suggest a Surface-associated Pathway for Iron Absorption in Schistosomes*. 
</div>

In [ ]:
%%time
df_temp = read_pubmed(TUTORIALS_ROOT / "Example_data/PMC1459476.xml", 
                      sim_cutoff=0.90 ,Print=False, Method = "Offline")
G = Plot_data(df_temp)

<div class="alert alert-block alert-success">
    <h2><b>Example 2 (Good):</b> PMC1802096</h2>
    MOR1/GEM1 plays an essential role in the plant-specific cytokinetic phragmoplast. 
</div>

In [ ]:
%%time
#####################################################
# Input method: Negative example
#####################################################
Out_def = read_pubmed(TUTORIALS_ROOT / "Example_data/PMC1802096.xml", 
                      Print=False, Method = "Offline", sim_cutoff=0.935)
Plot_data(Out_def)

In [ ]:
session_info.show(html=False)